# Module 07: Two-Stage Router Evaluation

This notebook evaluates a composed research policy with two explicit stages: Stage 1 selects GBDT/AutoGluon versus the TFM family; Stage 2 selects an eligible TFM if Stage 1 chooses that family. For each held-out dataset, Stage 2 development choices are learned from other development datasets. Stage 1 development family labels compare AutoGluon against the TFM selected without that label dataset. All seeds for one dataset stay in the same outer fold, and outer test outcomes are used only for final scoring.

The primary claim is predictive-quality family routing on this benchmark task mix. It does not validate user-preference understanding, explainability quality, or general speed superiority. The 20 task groups are exploratory in breadth; do not treat seeds as independent tasks. Run a pilot only to inspect data and execution health; use the frozen full protocol for paper estimates.


In [ ]:
# Install dependencies in the current Kaggle kernel. Restart the session after this cell.
import subprocess
import sys
from pathlib import Path

input_root = Path("/kaggle/input")
requirement_files = list(input_root.rglob("kaggle-requirements.txt")) if input_root.exists() else []
if not requirement_files:
    raise FileNotFoundError(
        "Couldn't find benchmark/kaggle-requirements.txt under /kaggle/input. "
        "Attach the private Module 07 source dataset."
)
# Prefer a requirements file beside the current module07 runner when duplicate datasets are attached.
requirement_files.sort(key=lambda p: ("module-7" not in str(p).casefold(), len(str(p))))
REQUIREMENTS_PATH = requirement_files[0]
print(f"Installing from: {REQUIREMENTS_PATH}", flush=True)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-r", str(REQUIREMENTS_PATH)],
    check=True,
)
print("\nDependency installation finished. Restart the Kaggle session now, then run the next cell.", flush=True)

## Source preflight and load runner

Run this cell after restarting the Kaggle session. It supports both a ZIP-only private dataset and datasets where Kaggle exposes the ZIP contents directly. The source bundle must contain the revised two-stage runner and matching protocol.


In [ ]:
import importlib.metadata as metadata
import importlib.util
import json
import zipfile
from pathlib import Path

INPUT_ROOT = Path("/kaggle/input")
WORK_ROOT = Path("/kaggle/working")
if not INPUT_ROOT.exists():
    raise FileNotFoundError("/kaggle/input is unavailable. Restart the Kaggle session and confirm the dataset is attached.")

# Find the actual runner path first; do not depend on cwd or import `benchmark` as a package.
runner_files = list(INPUT_ROOT.rglob("module07_runner.py"))
if not runner_files:
    bundles = list(INPUT_ROOT.rglob("module07_kaggle_source.zip"))
    if not bundles:
        raise FileNotFoundError(
            "No module07_runner.py or module07_kaggle_source.zip found under /kaggle/input. "
            "Attach the private Module 07 dataset."
        )
    extraction_root = WORK_ROOT / "module07_source"
    extraction_root.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(bundles[0]) as archive:
        archive.extractall(extraction_root)
    runner_files = list(extraction_root.rglob("module07_runner.py"))

valid = []
for path in runner_files:
    base = path.parent
    if (base / "module07_datasets.json").is_file() and (base / "kaggle-requirements.txt").is_file():
        valid.append(path)
if not valid:
    raise FileNotFoundError("Found a runner but not its matching manifest and pinned requirements file.")
valid.sort(key=lambda p: ("module-7" not in str(p).casefold(), len(str(p))))
RUNNER_PATH = valid[0]
SOURCE_DIR = RUNNER_PATH.parent
print(f"Runner: {RUNNER_PATH}")
print(f"Manifest: {SOURCE_DIR / 'module07_datasets.json'}")

EXPECTED_PACKAGES = {
    "numpy": "2.1.3",
    "pandas": "2.2.3",
    "pyarrow": "20.0.0",
    "scikit-learn": "1.6.1",
    "matplotlib": "3.10.0",
    "ucimlrepo": "0.0.7",
    "autogluon.tabular": "1.5.0",
    "tabpfn": "8.3.0",
    "tabicl": "2.2.0",
}
package_mismatches = []
for package, expected in EXPECTED_PACKAGES.items():
    try:
        installed = metadata.version(package)
    except metadata.PackageNotFoundError:
        installed = "missing"
    if installed != expected:
        package_mismatches.append(f"{package}: expected {expected}, found {installed}")
if package_mismatches:
    raise RuntimeError(
        "Pinned Kaggle dependencies are not active. "
        "Run the dependency installation cell, restart the session, and retry.\n"
        + "\n".join(package_mismatches)
    )
pyarrow_version = metadata.version("pyarrow")
import pyarrow as pa
if not hasattr(pa, "PyExtensionType"):
    raise RuntimeError("PyArrow's compiled module is incompatible or stale. Restart the Kaggle session, then rerun preflight.")

import torch
if not torch.cuda.is_available():
    raise RuntimeError("CUDA is unavailable. In Kaggle Notebook Settings, select a GPU accelerator, then restart the session.")
print(f"PyArrow: {pyarrow_version}; CUDA: {torch.cuda.get_device_name(0)}")

spec = importlib.util.spec_from_file_location("module07_runner", RUNNER_PATH)
if spec is None or spec.loader is None:
    raise ImportError(f"Couldn't create a Python loader for {RUNNER_PATH}")
runner = importlib.util.module_from_spec(spec)
spec.loader.exec_module(runner)
manifest = runner._load_manifest()
if len(manifest) != 20:
    raise RuntimeError(f"Expected the frozen 20-task manifest, found {len(manifest)} task groups.")
print(f"Ready: {len(manifest)} task groups; classification={sum(x['task'] == 'classification' for x in manifest)}; regression={sum(x['task'] == 'regression' for x in manifest)}")

## Dataset manifest and license records

The attached source bundle contains the 20-task manifest, dataset citations/license records, pinned dependency file, and runner. The runner validates targets against UCI metadata at runtime. Dataset rights are separate from model/checkpoint rights. Review the bundled `LICENSES_AND_CITATIONS.md` before the full run. TabPFN-3 has a separate model license and token requirement; TabICL checkpoint provenance is recorded by the runner.

Check `dataset_load_status.jsonl` and candidate failures before interpreting results. Adult, Bank Marketing, and Wine Quality have variant/label details that must be described accurately in any paper. Energy Efficiency uses Y1 (Heating Load) only; Y2 is excluded from predictors.

In [ ]:
# Pilot for execution health only; use the frozen full protocol for paper estimates.
import json
from pathlib import Path
RUN_MODE = "pilot"  # pilot: seed 17; full: seeds 17, 42, 2026
OUTPUT_DIR = Path("/kaggle/working/module07_artifacts")
AUTOGLUON_BUDGET_SECONDS = 120 if RUN_MODE == "pilot" else 300
print(f"Starting {RUN_MODE}; AutoGluon budget={AUTOGLUON_BUDGET_SECONDS}s per fit; output={OUTPUT_DIR}", flush=True)
run_status = runner.run_module07(
    OUTPUT_DIR,
    run_mode=RUN_MODE,
    auto_gluon_seconds=AUTOGLUON_BUDGET_SECONDS,
)
print(json.dumps(run_status, indent=2))

In [ ]:
# Inspect pilot health before considering the full run.
import pandas as pd
from IPython.display import display

attempts_path = OUTPUT_DIR / "raw_attempts.csv"
if attempts_path.exists():
    attempts = pd.read_csv(attempts_path)
    print("Candidate attempts by phase, engine, and status:")
    display(attempts.groupby(["phase", "engine", "status"], dropna=False).size().rename("count").reset_index())
    failures = attempts[attempts["status"].eq("failure")]
    print(f"Failed candidate attempts: {len(failures)}")
    if len(failures):
        display(failures[["dataset", "phase", "engine", "failure_type", "failure_message"]].head(50))
else:
    print("No completed attempt summary yet. If the run stopped, inspect dataset_load_status.jsonl and raw_attempts.jsonl in OUTPUT_DIR.")

load_status = OUTPUT_DIR / "dataset_load_status.jsonl"
if load_status.exists():
    loads = pd.read_json(load_status, lines=True)
    print("Dataset load status:")
    display(loads[[c for c in ["dataset", "status", "failure_type", "failure_message"] if c in loads.columns]])

for filename in ["route_family_counts.csv", "route_counts.csv", "stage1_development_labels.csv", "candidate_summary_by_dataset.csv", "task_level_metrics_summary.csv", "grouped_comparisons.csv"]:
    path = OUTPUT_DIR / filename
    if path.exists():
        print(filename)
        display(pd.read_csv(path))

## Read the outputs carefully

`stage1_development_labels` records GBDT-versus-TFM labels built from development validation outcomes. Each development dataset's TFM comparator is chosen from the other development datasets, so its own result does not choose its comparator. `router_decisions` records the held-out Stage 1 family choice, Stage 2 TFM choice, eligibility, and rationale. `outer_evaluation` scores the composed route on untouched test rows and compares it with AutoGluon-only and development-selected-TFM-only baselines. The outer-test best engine is descriptive oracle evidence only. Report classification and regression separately; aggregate seeds within dataset before uncertainty intervals. This study measures predictive-quality routing, not requirement-conformance, explainability quality, or universal speed advantages.


In [ ]:
# Package artifacts after a successful run. The zip is created in /kaggle/working for download.
import zipfile

zip_path = Path("/kaggle/working/module07_artifacts.zip")
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as archive:
    for path in OUTPUT_DIR.rglob("*"):
        if path.is_file():
            archive.write(path, path.relative_to(OUTPUT_DIR.parent))
print("Download the completed artifacts from:", zip_path)